## Importing libraries

In [0]:
from pyspark.sql import functions as F

## Reading Datasets

In [0]:
from_container = 'silver'
storage_acc = 'adlsolistchurn2026'
read_url = f'abfss://{from_container}@{storage_acc}.dfs.core.windows.net/'

to_container = 'gold'
write_url = f'abfss://{to_container}@{storage_acc}.dfs.core.windows.net/'

In [0]:
def report(df):
    df.show(5)
    print("*Schema*"*10)
    df.printSchema()
    print("*Distinct Count*"*10)
    print(df.count())
    print("*Null Count*"*10)
    print(df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).toPandas().to_string())



### olist_Customer 

In [0]:
olist_customers = spark.read.format('delta').load(read_url+'olist_customers')
olist_customers.show(5)

In [0]:
olist_geolocation = spark.read.format('delta').load(read_url+'olist_geolocation')
olist_geolocation.show(5)

In [0]:
olist_customers = olist_customers.alias('c').join(olist_geolocation.alias('g'), F.col('c.customer_zip_code_prefix')==F.col('g.geolocation_zip_code_prefix'),'left').select(['c.*','g.geolocation_lat','g.geolocation_lng'])

In [0]:
report(olist_customers)

In [0]:
olist_customers = olist_customers.withColumn(
    'geoloc_imputed',
    F.when(
        F.col('geolocation_lat').isNull()
        | F.col('geolocation_lng').isNull(),
        F.lit(True)
    ).otherwise(F.lit(False))
)


In [0]:
geo_lookup = olist_customers.groupBy('customer_city').agg(F.mode('geolocation_lat').alias('mode_geolocation_lat'), F.mode('geolocation_lng').alias('mode_geolocation_lng'))

In [0]:
olist_customers = olist_customers.alias('c').join(
    geo_lookup.alias('g'),
    on='customer_city',
    how='left'
).withColumn(
    'geolocation_lat',
    F.when(F.col('c.geolocation_lat').isNull(), F.col('g.mode_geolocation_lat')).otherwise(F.col('c.geolocation_lat'))
).withColumn(
    'geolocation_lng',
    F.when(F.col('c.geolocation_lng').isNull(), F.col('g.mode_geolocation_lng')).otherwise(F.col('c.geolocation_lng'))
).select('c.customer_id', 'c.customer_unique_id', 'c.customer_zip_code_prefix', 'c.customer_city', 'c.customer_state', 'geolocation_lat', 'geolocation_lng', 'geoloc_imputed')


In [0]:
brazil_states = {
    "SP": "São Paulo",
    "SC": "Santa Catarina",
    "MG": "Minas Gerais",
    "PR": "Paraná",
    "RJ": "Rio de Janeiro",
    "RS": "Rio Grande do Sul",
    "PA": "Pará",
    "GO": "Goiás",
    "ES": "Espírito Santo",
    "BA": "Bahia",
    "MA": "Maranhão",
    "MS": "Mato Grosso do Sul",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "RN": "Rio Grande do Norte",
    "PE": "Pernambuco",
    "MT": "Mato Grosso",
    "AM": "Amazonas",
    "AP": "Amapá",
    "AL": "Alagoas",
    "RO": "Rondônia",
    "PB": "Paraíba",
    "TO": "Tocantins",
    "PI": "Piauí",
    "AC": "Acre",
    "SE": "Sergipe",
    "RR": "Roraima"
}
olist_customers = olist_customers.replace(brazil_states,subset='customer_state')


In [0]:
olist_customers.write.mode('overwrite').format('delta').save(write_url+'olist_customers')

### olist_order_payments

In [0]:
olist_order_payments = spark.read.format('delta').load(read_url+'olist_order_payments')
olist_order_payments.show(5)

In [0]:
olist_order_payments.write.mode('overwrite').format('delta').save(write_url+'olist_order_payments')

### olist_order_reviews

In [0]:
olist_order_reviews = spark.read.format('delta').load(read_url+'olist_order_reviews')
olist_order_reviews.show(5)

In [0]:
olist_order_reviews.write.mode('overwrite').format('delta').save(write_url+'olist_order_reviews')

### olist_sellers

In [0]:
olist_sellers = spark.read.format('delta').load(read_url+'olist_sellers')
olist_sellers.show(5)

In [0]:
olist_sellers = olist_sellers.alias('s').join(olist_geolocation.alias('g'), F.col('s.seller_zip_code_prefix')==F.col('g.geolocation_zip_code_prefix'),'left').select(['s.*','g.geolocation_lat','g.geolocation_lng'])
report(olist_sellers)

In [0]:
olist_sellers = olist_sellers.withColumn(
    'geoloc_imputed',
    F.when(F.col('geolocation_lat').isNull(),F.lit(True)).otherwise(F.lit(False))
)


In [0]:
seller_geo_lookup = olist_sellers.groupBy('seller_city').agg(F.mode('geolocation_lat').alias('mode_geolocation_lat'),F.mode('geolocation_lng').alias('mode_geolocation_lng'))
seller_geo_lookup.show()

In [0]:
olist_sellers = olist_sellers.alias('s').join(
    seller_geo_lookup.alias('g'),
    on='seller_city',
    how='left'
).withColumn(
    'geolocation_lat',
    F.when(F.col('s.geolocation_lat').isNull(), F.col('g.mode_geolocation_lat')).otherwise(F.col('s.geolocation_lat'))
).withColumn(
    'geolocation_lng',
    F.when(F.col('s.geolocation_lng').isNull(), F.col('g.mode_geolocation_lng')).otherwise(F.col('s.geolocation_lng'))
).select('s.seller_id', 's.seller_zip_code_prefix', 's.seller_city', 's.seller_state', 'geolocation_lat', 'geolocation_lng', 'geoloc_imputed')


In [0]:
report(olist_sellers)

In [0]:
olist_sellers.write.format('delta').mode('overwrite').save(write_url+'olist_sellers')

### olist_products

In [0]:
olist_products = spark.read.format("delta").load(read_url+"olist_products")
olist_products.show(5)

In [0]:
product_category_name_translation = spark.read.format("delta").load(read_url+"product_category_name_translation")
product_category_name_translation.show(5)

In [0]:
olist_products = olist_products.alias('p').join(product_category_name_translation.alias('t'), F.col('p.product_category_name') == F.col('t.product_category_name'), 'left').select('p.product_id', 't.product_category_name_english', 'p.product_name_length', 'p.product_description_length', 'p.product_photos_qty', 'p.product_weight_g', 'p.product_length_cm', 'p.product_height_cm', 'p.product_width_cm', 'p.is_imputed')

In [0]:
report(olist_products)

In [0]:
olist_products.write.format('delta').mode('overwrite').save(write_url+'olist_products')

### olist_order_items

In [0]:
olist_order_items = spark.read.format('delta').load(read_url+'olist_order_items')
olist_order_items.show(5)

In [0]:
olist_order_items = olist_order_items. withColumn('total_price', F.round(F.col('price') + F.col('freight_value'),2))
olist_order_items.show(5)

In [0]:
olist_order_items.write.mode('overwrite').format('delta').save(write_url+'olist_order_items')

### olist_orders

In [0]:
olist_orders = spark.read.format('delta').load(read_url+'olist_orders')
olist_orders.show(5)

In [0]:
olist_orders = olist_orders.withColumn(
    'delivery_time_taken_days',
    F.datediff(F.col('order_delivered_customer_date'), F.col('order_purchase_timestamp'))
)


In [0]:
olist_orders.show(5)

In [0]:
order_items_temp = olist_order_items.groupBy('order_id').agg(F.count("product_id").alias('product_quantity'),
                                          F.sum("price").alias('total_price'),
                                          F.sum("freight_value").alias('total_freight_value'),
                                          F.sum("total_price").alias('total_cost'))

In [0]:
olist_order_records = olist_orders.alias('o').join(order_items_temp.alias('oi'), F.col('o.order_id') == F.col('oi.order_id'), 'left') \
    .drop("order_id")
olist_order_records = olist_order_records.fillna(0)

In [0]:
olist_order_records = olist_order_records.alias('oor').join(olist_customers.alias('oc'), olist_order_records.customer_id == olist_customers.customer_id, 'left').select('oor.*','oc.customer_unique_id', 'oc.customer_state')
olist_order_records.show(3)